In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.request import urlretrieve
from scipy import stats
from IPython.display import display
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [15]:
%pip install "xlrd>=2.0.1"

Note: you may need to restart the kernel to use updated packages.


In [16]:
for fn in ["Variable_dictionary.xls", "data/adsl.xls", "data/adacrm.xls"]:
    print(fn, pd.ExcelFile(fn).sheet_names)


vd = pd.read_excel("Variable_dictionary.xls", sheet_name="Dictionary")
adsl = pd.read_excel("data/adsl.xls", sheet_name="Sheet")
adacrm = pd.read_excel("data/adacrm.xls", sheet_name="Sheet")

print("IDs identical:", (adsl["subjid"] == adacrm["subjid"]).all())
print("Groups identical:", (adsl["group"] == adacrm["group"]).all())
print("\nRandomised per arm:\n", adacrm["group"].value_counts().sort_index())


Variable_dictionary.xls ['Dictionary']
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
data/adsl.xls ['Sheet']
data/adacrm.xls ['Sheet']
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
IDs identical: True
Groups identical: True

Randomised per arm:
 group
1    250
2    250
3    250
4    250
Name: count, dtype: int64


In [17]:
baseline_cols = ["age_round", "sex", "migzz_v1", "ech29_v1", "migpl1_v1", "migpl3_v1"]
print(adacrm.groupby("group")[baseline_cols].mean().round(2))

print("\nProtocol deviations by group:")
print(pd.crosstab(adsl["group"], adsl["pdtype"], dropna=False))

       age_round   sex  migzz_v1  ech29_v1  migpl1_v1  migpl3_v1
group                                                           
1          38.24  1.78      1.77      2.42       9.46       9.33
2          39.58  1.76      1.76      2.44      10.02      10.22
3          39.29  1.73      1.76      2.43      10.10      10.15
4          39.50  1.73      1.77      2.42      10.66      11.22

Protocol deviations by group:
pdtype  Failure to obtain primary endpoint  Informed consent withdrawal  \
group                                                                     
1                                        7                            3   
2                                        9                            6   
3                                        9                            5   
4                                       15                            2   

pdtype  Key inclusion/exclusion criteria violation  \
group                                                
1                   

In [18]:
# 1. Merge protocol deviations onto adacrm and subset to Aspirin (1) vs Metoprolol (4)
df = pd.merge(
    adsl[["subjid", "group", "pdtype"]],
    adacrm[["subjid", "fasfl", "pprotfl", "rrydl", "rrydllo"]],
    on="subjid",
)
sub = df[df["group"].isin([1, 4])].copy()

# 2. See which protocol deviations exclude participants from the FAS (fasfl == 2)
print("FAS inclusion by protocol deviation:")
print(pd.crosstab([sub["group"], sub["fasfl"]], sub["pdtype"].fillna("None")))

# 3. Within FAS == 1, see why rrydl is missing
fas_sub = sub[sub["fasfl"] == 1]
print("\nMissing week-12 outcome within FAS == 1:")
print(pd.crosstab([fas_sub["group"], fas_sub["rrydl"].isna()], fas_sub["pdtype"].fillna("None")))

FAS inclusion by protocol deviation:
pdtype       Failure to obtain primary endpoint  Informed consent withdrawal  \
group fasfl                                                                    
1     1                                       7                            0   
      2                                       0                            3   
4     1                                      15                            0   
      2                                       0                            2   

pdtype       None  non-compliance with the medication regimen  
group fasfl                                                    
1     1       238                                           2  
      2         0                                           0  
4     1       231                                           2  
      2         0                                           0  

Missing week-12 outcome within FAS == 1:
pdtype       Failure to obtain primary endpoint  None  \

In [19]:
for g, name in [(1, "Aspirin"), (4, "Metoprolol")]:
    arm = sub[sub["group"] == g]
    n_rand = len(arm)
    n_fas = (arm["fasfl"] == 1).sum()
    n_obs = ((arm["fasfl"] == 1) & (arm["rrydl"].notna())).sum()
    n_resp = ((arm["fasfl"] == 1) & (arm["rrydl"] == 1)).sum()
    print(f"{name}: Randomised={n_rand}, FAS={n_fas}, Observed Wk12={n_obs}, Responders={n_resp} ({n_resp / n_obs:.2%})")

Aspirin: Randomised=250, FAS=247, Observed Wk12=240, Responders=148 (61.67%)
Metoprolol: Randomised=250, FAS=248, Observed Wk12=233, Responders=144 (61.80%)


In [20]:
ana = sub[(sub["fasfl"] == 1) & (sub["rrydl"].notna())].copy()

# Recode 1=Yes -> 1, 2=No -> 0 explicitly (safer than (rrydl == 1).astype(int), which turns NaN into 0)
ana["response"] = ana["rrydl"].map({1: 1, 2: 0})

# Set Metoprolol (group 4) as reference (0) and Aspirin (group 1) as treatment (1)
ana["aspirin"] = (ana["group"] == 1).astype(int)

# Sanity check: confirm sample sizes and responder proportions match Step 2
print(ana.groupby("aspirin")["response"].agg(n_obs="count", responders="sum", prop="mean"))

         n_obs  responders      prop
aspirin                             
0          233         144  0.618026
1          240         148  0.616667


In [21]:
fit = smf.glm("response ~ aspirin", data=ana, family=sm.families.Binomial()).fit()
fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:               response   No. Observations:                  473
Model:                            GLM   Df Residuals:                      471
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -314.71
Date:                Mon, 23 Nov 2026   Deviance:                       629.42
Time:                        17:26:27   Pearson chi2:                     473.
No. Iterations:                     4   Pseudo R-squ. (CS):          1.954e-06
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.4812      0.135      3.569      0.000       0.217       0.745
aspirin       -0.0058      0.189     -0.030      0.976      -0.377       0.365
==============================================================================
"""

In [23]:
b0 = fit.params["Intercept"]
b1 = fit.params["aspirin"]
se_b1 = fit.bse["aspirin"]

or_est = np.exp(b1)
or_low, or_high = np.exp([b1 - 1.96 * se_b1, b1 + 1.96 * se_b1])
print(f"Odds Ratio: {or_est:.3f} (95% CI: {or_low:.3f} to {or_high:.3f})")

Odds Ratio: 0.994 (95% CI: 0.686 to 1.441)


In [24]:
p0 = 1 / (1 + np.exp(-b0))
p1 = 1 / (1 + np.exp(-(b0 + b1)))
diff = p1 - p0

n0 = (ana["aspirin"] == 0).sum()
n1 = (ana["aspirin"] == 1).sum()
se_diff = np.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)

diff_100 = 100 * diff
ci_low_100 = 100 * (diff - 1.96 * se_diff)
ci_high_100 = 100 * (diff + 1.96 * se_diff)
print(f"Difference per 100 treated: {diff_100:.2f} (95% CI: {ci_low_100:.2f} to {ci_high_100:.2f})")


Difference per 100 treated: -0.14 (95% CI: -8.90 to 8.63)


In [25]:
# Subset to all FAS participants in Aspirin (1) and Metoprolol (4)
fas_df = adacrm[(adacrm["group"].isin([1, 4])) & (adacrm["fasfl"] == 1)].copy()
fas_df["aspirin"] = (fas_df["group"] == 1).astype(int)

# 1. Sensitivity Check A: LOCF-imputed outcome (rrydllo) across all FAS (n = 247 vs 248)
fas_df["resp_locf"] = fas_df["rrydllo"].map({1: 1, 2: 0})
fit_locf = smf.glm("resp_locf ~ aspirin", data=fas_df, family=sm.families.Binomial()).fit()
or_locf = np.exp(fit_locf.params["aspirin"])
ci_locf = np.exp([
    fit_locf.params["aspirin"] - 1.96 * fit_locf.bse["aspirin"],
    fit_locf.params["aspirin"] + 1.96 * fit_locf.bse["aspirin"],
])
print(f"LOCF Sensitivity OR: {or_locf:.3f} (95% CI: {ci_locf[0]:.3f} to {ci_locf[1]:.3f})")

# 2. Sensitivity Check B: Covariate-adjusted model (accounting for baseline migraine days & aura)
ana_cov = fas_df[fas_df["rrydl"].notna()].copy()
ana_cov["response"] = ana_cov["rrydl"].map({1: 1, 2: 0})
fit_adj = smf.glm(
    "response ~ aspirin + migpl1_v1 + migzz_v1 + ech29_v1",
    data=ana_cov,
    family=sm.families.Binomial(),
).fit()
or_adj = np.exp(fit_adj.params["aspirin"])
ci_adj = np.exp([
    fit_adj.params["aspirin"] - 1.96 * fit_adj.bse["aspirin"],
    fit_adj.params["aspirin"] + 1.96 * fit_adj.bse["aspirin"],
])
print(f"Baseline-Adjusted Conditional OR: {or_adj:.3f} (95% CI: {ci_adj[0]:.3f} to {ci_adj[1]:.3f})")

# 3. Safety & Tolerability Check: Adverse events and bleeding rates (%) in the FAS
safety_cols = ["ae", "sae", "bleed", "hemor"]
safety_rates = fas_df.groupby("group")[safety_cols].apply(lambda g: (g == 1).mean() * 100).round(2)
print("\nSafety event rates (% of FAS, 1=Aspirin, 4=Metoprolol):")
print(safety_rates)

LOCF Sensitivity OR: 1.062 (95% CI: 0.742 to 1.520)
Baseline-Adjusted Conditional OR: 0.989 (95% CI: 0.681 to 1.437)

Safety event rates (% of FAS, 1=Aspirin, 4=Metoprolol):
          ae  sae  bleed  hemor
group                          
1      24.29  0.0  10.12    0.0
4      17.34  0.4   0.00    0.0
